# 04 - Model ComparisonCompare different models available through GoldenRay MaaS.## Models Available| Model | Runtime | Namespace | Notes ||-------|---------|-----------|-------|| TinyLlama | Ollama | demo-tenant | 1.1B params, CPU-friendly || facebook/opt-125m | vLLM | testuser | 125M params, very small || *Future: Llama-3-8B* | vLLM | — | Production quality || *Future: GPT-4o* | External | — | Via External Models gateway |

In [ ]:
import httpximport jsonimport timeimport osAPI_URL = os.getenv('GOLDENRAY_API_URL', 'http://10.0.0.113:30888')def call_model(namespace, endpoint, messages, max_tokens=200):    url = f'{API_URL}/api/v1/inference/{namespace}/{endpoint}'    start = time.time()    try:        resp = httpx.post(url, json={'messages': messages, 'max_tokens': max_tokens}, timeout=60)        latency = time.time() - start        data = resp.json()        text = data.get('choices', [{}])[0].get('message', {}).get('content', str(data))        return text, latency, None    except Exception as e:        return None, time.time() - start, str(e)models = [    ('demo-tenant', 'TinyLlama', 'TinyLlama 1.1B (Ollama)'),    ('testuser', 'facebook/opt-125m', 'OPT-125M (vLLM)'),]print('Models to compare:', [m[2] for m in models])

## Extraction Quality Comparison

In [ ]:
from pathlib import Pathclaim = Path('../data/sample_claims/claim4.txt').read_text()  # Simple claimmessages = [    {'role': 'system', 'content': 'Extract claim_id, claimant_name, claim_amount from the claim. Respond in JSON.'},    {'role': 'user', 'content': claim}]print('Test: Extract fields from claim4.txt (simple $2,400 auto claim)\n')for ns, endpoint, name in models:    text, latency, error = call_model(ns, endpoint, messages)    print(f'--- {name} ({latency:.1f}s) ---')    if error:        print(f'  Error: {error}')    else:        print(f'  {text[:300]}')    print()

## Key Takeaways1. **TinyLlama (1.1B)**: Can attempt extraction but output quality varies. Good for demos.2. **OPT-125M**: Too small for reliable structured extraction. Useful for testing infrastructure.3. **Larger models** (Llama-3-8B+, GPT-4o): Will produce reliable JSON extraction and classification.The application uses **rule-based fallbacks** so it works even when the LLM output is imperfect.The same API endpoint works for all models — just change `GOLDENRAY_NAMESPACE` and `GOLDENRAY_ENDPOINT`.